# Analisis Rantai Pasok & Peramalan Permintaan Ritel E-Commerce Indonesia

Notebook ini berfungsi sebagai mesin pemrosesan data (data pipeline), pemodelan statistik (SARIMAX), machine learning (Random Forest), serta penentuan parameter manajemen persediaan logistik (Safety Stock, ROP, EOQ) untuk operasional ritel e-commerce di Indonesia. 

Visualisasi hasil olahan data diekspor langsung ke folder `images/` dengan kualitas cetak tinggi (300 DPI) untuk diintegrasikan dalam laporan eksekutif di `README.md`.

### Objek Analisis:
* **Produk**: `SKU_HIJAB_INSTAN` (Produk fashion Muslim dengan volatilitas musiman sangat tinggi).
* **Cakupan Wilayah**: Transaksi agregat nasional dari 4 Gudang Regional utama (**Jakarta, Surabaya, Medan, Makassar**).
* **Variabel Eksternal**: Hari Libur Nasional Indonesia (Lebaran, Natal, Tahun Baru), Harbolnas (11.11 & 12.12), Variasi Harga (IDR), dan Gangguan Logistik Lokal (Banjir Musiman Jakarta & Kemacetan Pelabuhan Pra-Lebaran).


In [1]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

# Konfigurasi visualisasi profesional
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams.update({
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 14,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'figure.titlesize': 16,
    'figure.figsize': (12, 6)
})

# Path data & images
DATA_DIR = "./data"
IMAGES_DIR = "./images"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(IMAGES_DIR, exist_ok=True)
print("Pustaka berhasil diimpor & folder siap.")

Pustaka berhasil diimpor & folder siap.


In [2]:
# Set seed untuk konsistensi data
np.random.seed(42)

# Buat rentang tanggal harian (3 tahun: 2023-01-01 s/d 2025-12-31)
dates = pd.date_range(start="2023-01-01", end="2025-12-31", freq="D")
n_days = len(dates)

# Definisikan Gudang dan Produk
warehouses = ['Jakarta', 'Surabaya', 'Medan', 'Makassar']
products = {
    'SKU_HIJAB_INSTAN': {'category': 'Fashion', 'base_price': 65000, 'base_demand': 150, 'trend': 0.05},
    'SKU_KOPI_GAYO': {'category': 'Food & Beverage', 'base_price': 220000, 'base_demand': 80, 'trend': 0.02},
    'SKU_BATIK_TULIS': {'category': 'Fashion Premium', 'base_price': 1200000, 'base_demand': 15, 'trend': 0.005}
}

# Fungsi untuk menghitung tanggal Lebaran (Idul Fitri) di Indonesia (Simulasi Pendekatan)
# Lebaran: 2023 (22 April), 2024 (10 April), 2025 (30 Maret)
def get_days_to_lebaran(date):
    year = date.year
    if year == 2023:
        lebaran_date = pd.Timestamp("2023-04-22")
    elif year == 2024:
        lebaran_date = pd.Timestamp("2024-04-10")
    else:
        lebaran_date = pd.Timestamp("2025-03-30")
    
    diff = (lebaran_date - date).days
    return diff

# Simulasikan data harian di tingkat transaksi (Warehouse - Product - Date)
rows = []
for date in dates:
    # Faktor musiman kalender nasional
    days_to_leb = get_days_to_lebaran(date)
    
    # 1. Seasonality Lebaran (Puncak belanja busana muslim 3 minggu sebelum Lebaran)
    is_ramadan = 1 if (days_to_leb >= 0 and days_to_leb <= 25) else 0
    # Penurunan aktivitas logistik saat hari H Lebaran & mudik (Lebaran +/- 3 hari)
    is_lebaran_holiday = 1 if (abs(days_to_leb) <= 3) else 0
    
    # 2. Seasonality Harbolnas & Promo E-commerce (11.11, 12.12, Gajian akhir bulan)
    is_harbolnas = 1 if ((date.month == 11 and date.day == 11) or (date.month == 12 and date.day == 12)) else 0
    is_payday = 1 if (date.day >= 25 and date.day <= 28) else 0
    
    # 3. Seasonality Akhir Tahun (Natal & Tahun Baru)
    is_year_end = 1 if (date.month == 12 and date.day >= 20) else 0
    
    # 4. Gangguan Logistik Lokal (Faktor Cuaca/Banjir Jakarta di Jan-Feb & puncak macet pelabuhan sebelum Lebaran)
    # Banjir Jakarta terjadi secara acak di Jan-Feb dengan probabilitas tinggi
    is_jakarta_flood = 1 if (date.month in [1, 2] and np.random.rand() < 0.08) else 0
    # Kemacetan Pelabuhan pra-Lebaran (7 hari sebelum Lebaran)
    is_shipping_congestion = 1 if (days_to_leb > 0 and days_to_leb <= 7) else 0
    
    for wh in warehouses:
        # Bobot volume gudang regional
        wh_weight = {'Jakarta': 1.0, 'Surabaya': 0.6, 'Medan': 0.4, 'Makassar': 0.2}[wh]
        
        for prod, p_info in products.items():
            # Trend harian
            day_idx = (date - dates[0]).days
            trend_val = p_info['trend'] * day_idx
            
            # Base demand + trend
            demand = p_info['base_demand'] + trend_val
            
            # Seasonality Mingguan (Jumat - Minggu naik untuk Fashion, Kopi stabil)
            dayofweek = date.dayofweek
            if p_info['category'] == 'Fashion':
                weekly_effect = {0: -10, 1: -5, 2: 0, 3: 5, 4: 15, 5: 35, 6: 25}[dayofweek]
            else:
                weekly_effect = {0: -2, 1: -1, 2: 0, 3: 2, 4: 5, 5: 8, 6: 6}[dayofweek]
            demand += weekly_effect
            
            # Efek Lebaran (sangat kuat untuk Hijab dan Batik Premium)
            if is_ramadan:
                ramadan_mult = {'SKU_HIJAB_INSTAN': 2.2, 'SKU_BATIK_TULIS': 1.8, 'SKU_KOPI_GAYO': 1.15}[prod]
                demand *= ramadan_mult
            elif is_lebaran_holiday:
                # Penjualan drop karena ekspedisi libur total
                demand *= 0.15
                
            # Efek Harbolnas & Akhir Tahun
            if is_harbolnas:
                demand += {'SKU_HIJAB_INSTAN': 120, 'SKU_BATIK_TULIS': 30, 'SKU_KOPI_GAYO': 50}[prod]
            elif is_year_end:
                demand += {'SKU_HIJAB_INSTAN': 30, 'SKU_BATIK_TULIS': 10, 'SKU_KOPI_GAYO': 20}[prod]
            elif is_payday:
                demand += {'SKU_HIJAB_INSTAN': 15, 'SKU_BATIK_TULIS': 3, 'SKU_KOPI_GAYO': 8}[prod]
                
            # Skema Promosi & Penurunan Harga
            # Promosi E-Commerce di Indonesia sangat aktif saat Harbolnas & Gajian
            is_promo = 1 if (is_harbolnas or (is_payday and np.random.rand() < 0.4) or (is_ramadan and np.random.rand() < 0.3)) else 0
            price = p_info['base_price']
            if is_promo:
                # Diskon 20%
                price = int(price * 0.8)
                # Kenaikan demand karena diskon harga
                demand *= 1.45
                
            # Tambahkan noise acak
            noise = np.random.normal(0, demand * 0.08)
            demand_raw = int(np.clip(demand * wh_weight + noise, 2, None))
            
            # Simulasi Stockout (Masalah Logistik & Distribusi Indonesia)
            # 1. Banjir di Jakarta berdampak pada Gudang Jakarta
            # 2. Kemacetan Pelabuhan pra-Lebaran berdampak pada Makassar & Medan (luar Jawa)
            stockout = 0
            if wh == 'Jakarta' and is_jakarta_flood and np.random.rand() < 0.5:
                stockout = 1
            elif wh in ['Makassar', 'Medan'] and is_shipping_congestion and np.random.rand() < 0.4:
                stockout = 1
            elif np.random.rand() < 0.02: # Kendala kurir acak harian
                stockout = 1
                
            sales_actual = 0 if stockout == 1 else demand_raw
            
            rows.append({
                'Date': date,
                'Warehouse': wh,
                'Product': prod,
                'Category': p_info['category'],
                'Price_IDR': price,
                'Promotion': is_promo,
                'Demand_Raw': demand_raw,
                'Stockout_Occurred': stockout,
                'Sales_Actual': sales_actual
            })

df_raw = pd.DataFrame(rows)

# Simpan data mentah transaksional
raw_file_path = os.path.join(DATA_DIR, "raw_demand.csv")
df_raw.to_csv(raw_file_path, index=False)
print(f"Data transaksi mentah ritel Indonesia disimulasikan: {df_raw.shape[0]} baris.")
df_raw.head()

Data transaksi mentah ritel Indonesia disimulasikan: 13152 baris.


,Date,Warehouse,Product,Category,Price_IDR,Promotion,Demand_Raw,Stockout_Occurred,Sales_Actual
0,2023-01-01,Jakarta,SKU_HIJAB_INSTAN,Fashion,65000,0,159,0,159
1,2023-01-01,Jakarta,SKU_KOPI_GAYO,Food & Beverage,220000,0,88,0,88
2,2023-01-01,Jakarta,SKU_BATIK_TULIS,Fashion Premium,1200000,0,21,0,21
3,2023-01-01,Surabaya,SKU_HIJAB_INSTAN,Fashion,65000,0,119,0,119
4,2023-01-01,Surabaya,SKU_KOPI_GAYO,Food & Beverage,220000,0,47,0,47


### Prapemrosesan Data (Imputasi Stockout Nasional)

Untuk peramalan tingkat nasional, kita akan mengagregasikan data harian untuk produk target utama kita: `SKU_HIJAB_INSTAN` di seluruh regional gudang. 

Gangguan pengiriman menyebabkan kegagalan penjualan (*stockout*). Kita harus mendeteksi hari-hari di mana terjadi gangguan logistik lokal dan melakukan imputasi untuk mendapatkan estimasi kebutuhan pasar riil (*unconstrained demand*).


In [3]:
# Filter data untuk SKU_HIJAB_INSTAN
df_hijab = df_raw[df_raw['Product'] == 'SKU_HIJAB_INSTAN'].copy()

# Agregasikan ke tingkat harian nasional (Sum Demand & Sales, Average Price & Promotion)
df_national = df_hijab.groupby('Date').agg({
    'Price_IDR': 'mean',
    'Promotion': 'max',
    'Demand_Raw': 'sum',
    'Sales_Actual': 'sum',
    'Stockout_Occurred': 'max'  # Indikator jika ada minimal 1 gudang yang mengalami stockout
}).reset_index()

# Konversi tipe data agar rapi
df_national['Price_IDR'] = df_national['Price_IDR'].round().astype(int)

# Salin kolom penjualan aktual ke kolom target permintan bersih (Demand_Cleaned)
df_national['Demand_Cleaned'] = df_national['Sales_Actual']

# Deteksi hari di mana terdapat stockout dan penjualan aktual di bawah demand riil
# Kita lakukan imputasi menggunakan rata-rata bergerak 7 hari sebelumnya untuk memulihkan permintaan asli
rolling_impute = df_national['Sales_Actual'].rolling(window=7, min_periods=1, closed='left').mean()
df_national.loc[df_national['Stockout_Occurred'] == 1, 'Demand_Cleaned'] = rolling_impute

# Pastikan tipe data bulat dan tidak ada NaN
df_national['Demand_Cleaned'] = df_national['Demand_Cleaned'].fillna(method='bfill').round().astype(int)

# Simpan data bersih
cleaned_file_path = os.path.join(DATA_DIR, "cleaned_demand.csv")
df_national.to_csv(cleaned_file_path, index=False)
print("Data agregat nasional bersih untuk SKU_HIJAB_INSTAN berhasil disimpan.")
df_national[df_national['Stockout_Occurred'] == 1].head()

Data agregat nasional bersih untuk SKU_HIJAB_INSTAN berhasil disimpan.


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_15604\179601867.py:22: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[ 335.          261.5         334.71428571  360.85714286  370.42857143
  358.85714286  347.71428571  782.71428571  858.          770.14285714
  860.28571429  720.85714286  698.          692.42857143  756.85714286
  562.42857143  353.71428571  376.85714286  352.71428571  364.14285714
  376.85714286  392.57142857  401.42857143  353.71428571  368.
  398.14285714  449.85714286  384.28571429  390.85714286  468.57142857
  398.57142857  364.57142857  393.85714286  403.57142857  372.14285714
  396.57142857  938.14285714 1016.71428571  976.28571429  944.28571429
  859.28571429  451.57142857  459.14285714  424.          401.57142857
  404.71428571  448.28571429  411.71428571  371.71428571  418.85714286
  439.          417.57142857  395.          496.          395.85714286
  441.42857143  514.28571429  

,Date,Price_IDR,Promotion,Demand_Raw,Sales_Actual,Stockout_Occurred,Demand_Cleaned
1,2023-01-02,65000,0,315,188,1,335
2,2023-01-03,65000,0,331,281,1,262
7,2023-01-08,65000,0,402,328,1,335
14,2023-01-15,65000,0,341,275,1,361
42,2023-02-12,65000,0,385,336,1,370


### Eksplorasi Data (EDA) & Analisis Musiman Indonesia

Bagian ini mengeksplorasi grafik tren permintaan serta menguraikan komponen musiman mingguan. Kita dapat melihat dengan jelas efek lonjakan musiman tahunan saat menjelang Lebaran (Idul Fitri) dan promosi Harbolnas akhir tahun.


In [4]:
# 1. Plot Tren Permintaan & Penjualan Nasional
plt.figure(figsize=(12, 6))
plt.plot(df_national['Date'], df_national['Demand_Cleaned'], label='Permintaan Pasar Riil (Imputasi)', color='#2980b9', alpha=0.85, linewidth=1.5)
plt.plot(df_national['Date'], df_national['Sales_Actual'], label='Penjualan Aktual Tercatat (Stockout Terjadi)', color='#e67e22', alpha=0.4, linestyle=':')

# Tandai periode Lebaran untuk memandu interpretasi grafik
# Lebaran 2023 (April 22), 2024 (April 10), 2025 (Maret 30)
lebaran_dates = [pd.Timestamp("2023-04-22"), pd.Timestamp("2024-04-10"), pd.Timestamp("2025-03-30")]
for i, l_date in enumerate(lebaran_dates):
    plt.axvline(x=l_date, color='#c0392b', linestyle='--', alpha=0.6, label='Hari Raya Lebaran' if i==0 else "")
    plt.text(l_date - pd.Timedelta(days=40), plt.gca().get_ylim()[1]*0.8, 'Puncak Lebaran', color='#c0392b', fontsize=9, fontweight='bold')

plt.title('Tren Permintaan SKU_HIJAB_INSTAN Nasional di Indonesia (2023-2025)', fontsize=14, fontweight='bold')
plt.xlabel('Tanggal', fontsize=12)
plt.ylabel('Kuantitas Unit', fontsize=12)
plt.legend(loc='upper left', frameon=True)
plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, "demand_trend.png"), dpi=300)
plt.close()

# 2. Dekomposisi Musiman Mingguan (Period=7)
df_ts = df_national.set_index('Date')
decomposition = seasonal_decompose(df_ts['Demand_Cleaned'], model='additive', period=7)

fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)
decomposition.observed.plot(ax=axes[0], color='#2c3e50', legend=False)
axes[0].set_ylabel('Observed')
axes[0].set_title('Dekomposisi Aditif Tren & Musiman Permintaan Bersih (Weekly Period=7)', fontsize=14, fontweight='bold')

decomposition.trend.plot(ax=axes[1], color='#c0392b', legend=False)
axes[1].set_ylabel('Trend (Tren Ritel)')

decomposition.seasonal.plot(ax=axes[2], color='#8e44ad', legend=False)
axes[2].set_ylabel('Weekly Seasonality')

decomposition.resid.plot(ax=axes[3], color='#7f8c8d', style='.', legend=False)
axes[3].set_ylabel('Residual (Noise)')
axes[3].set_xlabel('Tanggal')

plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, "seasonal_decomposition.png"), dpi=300)
plt.close()

print("Grafik visualisasi ritel Indonesia berhasil disimpan.")

Grafik visualisasi ritel Indonesia berhasil disimpan.


### Feature Engineering untuk Model Machine Learning

Kita membuat fitur lag penjualan historis ($t-1, t-7, t-14, t-30$), fitur statistik berbasis jendela bergerak (*rolling mean*), dan data kalender untuk menangkap pola temporal bulanan dan mingguan.


In [5]:
df_features = df_national.copy()

# Buat fitur lags (mundur waktu)
df_features['lag_1'] = df_features['Demand_Cleaned'].shift(1)
df_features['lag_7'] = df_features['Demand_Cleaned'].shift(7)
df_features['lag_14'] = df_features['Demand_Cleaned'].shift(14)
df_features['lag_30'] = df_features['Demand_Cleaned'].shift(30)

# Buat fitur rolling mean (rata-rata bergerak)
df_features['rolling_mean_7'] = df_features['Demand_Cleaned'].shift(1).rolling(window=7).mean()
df_features['rolling_mean_30'] = df_features['Demand_Cleaned'].shift(1).rolling(window=30).mean()

# Buat fitur temporal berbasis kalender
df_features['day_of_week'] = df_features['Date'].dt.dayofweek
df_features['month'] = df_features['Date'].dt.month
df_features['year'] = df_features['Date'].dt.year
df_features['is_weekend'] = df_features['day_of_week'].isin([5, 6]).astype(int)

# Hapus nilai NaN akibat shift / rolling window
df_features = df_features.dropna().reset_index(drop=True)
print(f"Dimensi data setelah feature engineering: {df_features.shape}")
df_features.head(2)

Dimensi data setelah feature engineering: (1066, 17)


,Date,Price_IDR,Promotion,Demand_Raw,Sales_Actual,Stockout_Occurred,Demand_Cleaned,lag_1,lag_7,lag_14,lag_30,rolling_mean_7,rolling_mean_30,day_of_week,month,year,is_weekend
0,2023-01-31,65000,0,288,288,0,288,273.0,346.0,354.0,335.0,387.285714,361.333333,1,1,2023,0
1,2023-02-01,65000,0,319,319,0,319,288.0,343.0,358.0,335.0,379.000000,359.766667,2,2,2023,0


### Pemodelan Peramalan Permintaan Ritel

Untuk mengevaluasi performa model secara objektif, kita membagi data menjadi dua bagian:
- **Training Set**: Periode 2023-01-01 hingga 2025-06-30 (untuk melatih model).
- **Testing Set (Holdout)**: Periode 2025-07-01 hingga 2025-12-31 (184 hari untuk pengujian performa prediksi).

Model yang diuji:
1. **Seasonal Naive (Baseline)**: Model acuan yang menebak penjualan hari ini sama dengan penjualan 7 hari lalu (`lag_7`).
2. **SARIMAX**: Model statistik linear parametrik yang memperhitungkan regresi variabel eksogen harga (`Price_IDR`) dan promosi (`Promotion`) serta struktur autokorelasi musiman mingguan.
3. **Random Forest Regressor**: Algoritma machine learning non-parametrik yang mampu menangkap hubungan non-linear kompleks antara lag, tren, harga, promosi, dan kalender secara fleksibel.


In [6]:
# Split Data Train/Test
split_date = pd.to_datetime("2025-07-01")
train_df = df_features[df_features['Date'] < split_date].copy()
test_df = df_features[df_features['Date'] >= split_date].copy()

# 1. Baseline Model (Seasonal Naive)
test_df['Forecast_Baseline'] = test_df['lag_7']

# 2. SARIMAX Model
# Tarik variabel eksogen
y_train_sari = train_df['Demand_Cleaned'].values
exog_train_sari = train_df[['Price_IDR', 'Promotion']].values
exog_test_sari = test_df[['Price_IDR', 'Promotion']].values

# Fit SARIMAX(1, 1, 1)x(1, 0, 0, 7)
sarimax_model = SARIMAX(
    y_train_sari,
    exog=exog_train_sari,
    order=(1, 1, 1),
    seasonal_order=(1, 0, 0, 7),
    enforce_stationarity=False,
    enforce_invertibility=False
)
sarimax_fit = sarimax_model.fit(disp=False)

# Forecast out-of-sample
test_df['Forecast_SARIMAX'] = sarimax_fit.forecast(steps=len(test_df), exog=exog_test_sari)

# 3. Random Forest Regressor
features = ['lag_1', 'lag_7', 'lag_14', 'lag_30', 'rolling_mean_7', 'rolling_mean_30', 
            'Price_IDR', 'Promotion', 'day_of_week', 'month', 'is_weekend']

X_train_rf = train_df[features]
y_train_rf = train_df['Demand_Cleaned']
X_test_rf = test_df[features]

rf_model = RandomForestRegressor(n_estimators=150, random_state=42, n_jobs=-1)
rf_model.fit(X_train_rf, y_train_rf)

test_df['Forecast_RF'] = rf_model.predict(X_test_rf)
print("Seluruh model berhasil dilatih.")

Seluruh model berhasil dilatih.


### Perbandingan Kinerja Prediction Model

Kita menghitung performa akurasi peramalan menggunakan tiga metrik evaluasi standar:
- **MAE (Mean Absolute Error)**: Mengukur rata-rata besaran absolut kesalahan prediksi dalam unit produk.
- **RMSE (Root Mean Squared Error)**: Memberi bobot lebih tinggi pada kesalahan besar.
- **MAPE (Mean Absolute Percentage Error %)**: Menilai akurasi relatif kesalahan dalam bentuk persentase terhadap aktual.


In [7]:
# Fungsi Perhitungan MAPE
def calculate_mape(actual, forecast):
    return np.mean(np.abs((actual - forecast) / actual)) * 100

y_actual = test_df['Demand_Cleaned'].values
models_list = ['Baseline', 'SARIMAX', 'RF']
metrics = {}

for m in models_list:
    col = f'Forecast_{m}'
    pred = test_df[col].values
    mae = mean_absolute_error(y_actual, pred)
    rmse = root_mean_squared_error(y_actual, pred)
    mape = calculate_mape(y_actual, pred)
    metrics[m] = {'MAE': round(mae, 2), 'RMSE': round(rmse, 2), 'MAPE %': round(mape, 2)}

df_metrics = pd.DataFrame(metrics).T
print("TABEL PERBANDINGAN METRIK KINERJA MODEL:")
print(df_metrics)

# Ekspor tabel metrik ke CSV untuk kemudahan pelaporan di README
df_metrics.to_csv(os.path.join(DATA_DIR, "metrics_comparison.csv"), index=True)

# Plot Perbandingan Prediksi vs Aktual (Fokus pada 60 hari terakhir 2025 agar grafik lebih terbaca)
plot_df = test_df.tail(60).copy()

plt.figure(figsize=(12, 6))
plt.plot(plot_df['Date'], plot_df['Demand_Cleaned'], label='Permintaan Aktual (Bersih)', color='#2c3e50', linewidth=2, marker='o')
plt.plot(plot_df['Date'], plot_df['Forecast_Baseline'], label='Baseline (Seasonal Naive)', color='#7f8c8d', linestyle='--', alpha=0.6)
plt.plot(plot_df['Date'], plot_df['Forecast_SARIMAX'], label='SARIMAX (Statistik)', color='#d35400', linestyle='-.', alpha=0.85)
plt.plot(plot_df['Date'], plot_df['Forecast_RF'], label='Random Forest (Machine Learning)', color='#27ae60', linewidth=1.5, alpha=0.9)

plt.title('Perbandingan Model Peramalan vs Permintaan Aktual SKU_HIJAB_INSTAN (60 Hari Terakhir 2025)', fontsize=13, fontweight='bold')
plt.xlabel('Tanggal', fontsize=11)
plt.ylabel('Permintaan Unit Ritel', fontsize=11)
plt.legend(loc='upper left', frameon=True)
plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, "forecast_vs_actual.png"), dpi=300)
plt.close()

# Plot Feature Importance Random Forest
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]
features_sorted = [features[i] for i in indices]

plt.figure(figsize=(10, 5))
sns.barplot(x=importances[indices], y=features_sorted, hue=features_sorted, legend=False, palette='viridis')
plt.title('Fitur Paling Berpengaruh (Feature Importance) - Model Random Forest', fontsize=13, fontweight='bold')
plt.xlabel('Skor Kepentingan Relatif', fontsize=11)
plt.ylabel('Fitur', fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, "feature_importance.png"), dpi=300)
plt.close()

print("Semua grafik model dan metrik evaluasi telah diekspor.")

TABEL PERBANDINGAN METRIK KINERJA MODEL:
            MAE    RMSE  MAPE %
Baseline  73.01  115.34   14.37
SARIMAX   47.89   64.89    9.30
RF        40.76   53.67    8.00


Semua grafik model dan metrik evaluasi telah diekspor.


### Optimasi Inventaris (Safety Stock, ROP, & EOQ)

Menggunakan prediksi dari model peramalan terbaik kita (Random Forest), kita dapat menghitung parameter operasional logistik gudang ritel Indonesia:
1. **Safety Stock (SS)**: Stok pengaman untuk meredam fluktuasi permintaan tak terduga serta keterlambatan pengiriman logistik antar-pulau.
   $$SS = Z \times \sigma_e \times \sqrt{L}$$
   * Di mana $Z = 1.645$ (tingkat keandalan layanan / service level 95%), $L = 5$ hari (Lead Time rata-rata pengiriman domestik), dan $\sigma_e$ adalah standar deviasi dari error residual prediksi Random Forest.
2. **Reorder Point (ROP)**: Batas tingkat persediaan minimum yang memicu pemesanan baru ke produsen/distributor.
   $$ROP = (d \times L) + SS$$
   * Di mana $d$ adalah rata-rata tingkat permintaan harian nasional.
3. **Economic Order Quantity (EOQ)**: Jumlah kuantitas pemesanan sekali pesan yang paling optimal secara biaya, guna meminimalkan total penjumlahan biaya pemesanan (*ordering cost*) dan biaya penyimpanan gudang (*holding cost*).
   $$EOQ = \sqrt{\frac{2DS}{H}}$$
   * Di mana $D$ adalah total permintaan tahunan, $S$ adalah biaya administrasi & pengiriman sekali pesan, dan $H$ adalah biaya penyimpanan inventaris per unit per tahun.


In [8]:
# 1. Parameter Persediaan
Z = 1.645  # 95% Service Level (Standar Operasional Retail)
L = 5      # Lead Time pengiriman barang dari pabrik/vendor (5 hari)

# Hitung standar deviasi kesalahan (residual) model peramalan terbaik (Random Forest)
residuals = test_df['Demand_Cleaned'] - test_df['Forecast_RF']
sigma_e = residuals.std()

d = test_df['Demand_Cleaned'].mean() # Rata-rata permintaan harian nasional di periode test
D = d * 365 # Proyeksi permintaan tahunan (~100.000+ unit)

# Biaya Operasional Domestik Indonesia
S = 750000  # Biaya pengiriman & pemesanan logistik per order (IDR 750.000)
H = 12000   # Biaya sewa gudang & penyimpanan per unit per tahun (IDR 12.000)

# 2. Hitung Formulasi Logistik
safety_stock = int(np.round(Z * sigma_e * np.sqrt(L)))
reorder_point = int(np.round((d * L) + safety_stock))
eoq = int(np.round(np.sqrt((2 * D * S) / H)))

# Simpan metrik logistik ke JSON untuk diakses dalam README
inventory_metrics = {
    'Average_Daily_Demand': round(d, 2),
    'Residual_Std_Dev_Units': round(sigma_e, 2),
    'Safety_Stock_Units': safety_stock,
    'Reorder_Point_Units': reorder_point,
    'Economic_Order_Quantity_Units': eoq
}
with open(os.path.join(DATA_DIR, "inventory_metrics.json"), 'w') as f:
    json.dump(inventory_metrics, f, indent=4)

print("HASIL KALKULASI PARAMETER MANAJEMEN INVENTARIS:")
for k, v in inventory_metrics.items():
    print(f"* {k.replace('_', ' ')}: {v}")

# 3. Jalankan Simulasi Tingkat Persediaan Harian (Inventory Sawtooth Simulation)
current_inv = eoq + safety_stock
order_in_transit = False
days_to_arrival = 0

inventory_levels = []
order_dates = []
arrival_dates = []

dates_sim = test_df['Date'].values
demand_sim = test_df['Demand_Cleaned'].values

for i in range(len(test_df)):
    # Hari baru dimulai, periksa pesanan dalam perjalanan
    if order_in_transit:
        days_to_arrival -= 1
        if days_to_arrival == 0:
            current_inv += eoq
            order_in_transit = False
            arrival_dates.append(dates_sim[i])
            
    # Permintaan hari ini dipenuhi dari stok
    current_inv = max(0, current_inv - demand_sim[i])
    inventory_levels.append(current_inv)
    
    # Periksa batas reorder point (ROP) untuk melakukan pemesanan baru
    if current_inv <= reorder_point and not order_in_transit:
        order_in_transit = True
        days_to_arrival = L
        order_dates.append(dates_sim[i])

test_df['Inventory_Level'] = inventory_levels

# Plot Grafik Persediaan Gergaji (Inventory Sawtooth Plot)
plt.figure(figsize=(12, 6))
plt.plot(test_df['Date'], test_df['Inventory_Level'], label='Tingkat Persediaan Fisik Gudang', color='#1abc9c', linewidth=2)
plt.axhline(y=reorder_point, color='#e67e22', linestyle='--', label=f'Reorder Point (ROP = {reorder_point} unit)', alpha=0.8)
plt.axhline(y=safety_stock, color='#e74c3c', linestyle=':', label=f'Safety Stock (SS = {safety_stock} unit)', alpha=0.8)

# Arsir area zona resiko stockout di bawah safety stock
plt.fill_between(test_df['Date'], 0, safety_stock, color='#c0392b', alpha=0.1, label='Zona Risiko Kehabisan Stok (Stockout)')

# Tandai titik pemesanan baru (Order Placed)
if order_dates:
    plt.scatter(order_dates, [reorder_point] * len(order_dates), color='#d35400', marker='v', s=80, zorder=5, label='Pemesanan Baru (Kirim Rilis EOQ)')

plt.title('Simulasi Level Persediaan Gudang SKU_HIJAB_INSTAN dengan ROP & EOQ (Semester II 2025)', fontsize=13, fontweight='bold')
plt.xlabel('Tanggal', fontsize=11)
plt.ylabel('Tingkat Stok Inventaris (Unit)', fontsize=11)
plt.legend(loc='upper right', frameon=True)
plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, "safety_stock_rop.png"), dpi=300)
plt.close()

print("Grafik simulasi persediaan safety stock & ROP berhasil disimpan.")

HASIL KALKULASI PARAMETER MANAJEMEN INVENTARIS:
* Average Daily Demand: 483.21
* Residual Std Dev Units: 49.17
* Safety Stock Units: 181
* Reorder Point Units: 2597
* Economic Order Quantity Units: 4695


Grafik simulasi persediaan safety stock & ROP berhasil disimpan.
